# 01 · BFCL Reproduction on a Kaggle T4

**Project:** Silent Tool-Call Failures: How Quantization Degrades Function Calling in Small LLMs
**Update 2 goal:** show that the Berkeley Function Calling Leaderboard (BFCL) harness runs on Kaggle's free T4 GPU in FP16, and that its scores match the published leaderboard. Every later experiment builds on this check.

**Model:** `Qwen/Qwen3-4B-Instruct-2507`, in both prompt mode and native function-calling (FC) mode.
BFCL v4 no longer supports Qwen2.5 (retired in July 2025), so a Qwen2.5 score cannot be compared with the current leaderboard. Qwen3-4B-Instruct-2507 is on the leaderboard in both modes, does not use "thinking" mode, and fits on one T4 in FP16.

**Benchmark:** BFCL `non_live` (1,390 questions): simple (Python, Java, JavaScript), multiple, parallel, parallel-multiple and irrelevance.

## How to run on Kaggle

1. kaggle.com → **Create → New Notebook → File → Import Notebook**, then upload this file.
2. In the right panel, under **Settings**: **Accelerator → GPU T4 x2** and **Internet → On**.
3. Optional for Qwen, required later for Llama: **Add-ons → Secrets** → add `HF_TOKEN` (your Hugging Face read token) and switch it on for this notebook.
4. Click **Save Version → Save & Run All (Commit)**. It runs in the background, so you can close the tab. To watch it live instead, use **Run All**.

**Time:** about 40 minutes in total, including setup. That uses about 1 of your 30 weekly GPU-hours.
**Output:** `bfcl_repro_results.zip` contains scores, the leaderboard comparison, `bfcl_repro.md`, run information and logs.

Each step checks its own result and stops with a clear message if something is wrong.

## 1. Settings
Edit this cell only.

In [1]:
import os

# BFCL model names to run. All of them must use the same Hugging Face model (SERVE_MODEL),
# because one vLLM server answers for all of them.
#   no suffix = prompt mode, "-FC" = native function-calling mode
MODELS = [
    "Qwen/Qwen3-4B-Instruct-2507",
    "Qwen/Qwen3-4B-Instruct-2507-FC",
]
SERVE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"

TEST_CATEGORY = "non_live"    # 7 categories, 1,390 questions

BFCL_VERSION = "2026.3.23"    # pinned for reproducibility
DTYPE = "half"                # FP16. T4 GPUs do not support BF16.
MAX_MODEL_LEN = 16384         # fits every non-live prompt plus BFCL's 4,096-token answer budget
GPU_MEMORY_UTILIZATION = 0.90
TENSOR_PARALLEL_SIZE = 1      # use 2 (both T4s) for models that do not fit on one T4, e.g. 7-8B in FP16
PORT = 1053                   # BFCL's default local server port

WORK = "/kaggle/working/bfcl"                 # results, scores and logs (kept as notebook output)
VENV = os.path.expanduser("~/bfcl-venv")      # separate Python 3.11 environment (not kept)

MODELS_ARG = ",".join(MODELS)

## 2. Check the GPU and internet, and set environment variables

In [2]:
import json
import shutil
import subprocess
import time
from datetime import date

import requests

# GPU
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode != 0:
    raise RuntimeError("No GPU found. Set Settings -> Accelerator -> GPU T4 x2, then run again.")
print("GPUs:\n" + smi.stdout)

# Internet
try:
    requests.get("https://pypi.org/simple/", timeout=15).raise_for_status()
    print("Internet: OK")
except Exception as e:
    raise RuntimeError("No internet. Set Settings -> Internet -> On (needs a phone-verified Kaggle account).") from e

# Settings read by BFCL and vLLM
os.makedirs(WORK, exist_ok=True)
os.environ["BFCL_PROJECT_ROOT"] = WORK        # BFCL writes result/ and score/ here
os.environ["LOCAL_SERVER_PORT"] = str(PORT)   # BFCL sends its requests to this port
os.environ["VLLM_USE_V1"] = "0"               # T4 (compute capability 7.5) runs on vLLM's V0 engine

# Hugging Face token: optional for Qwen, required for gated models such as Llama 3.2
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: loaded from Kaggle Secrets")
except Exception:
    print("HF_TOKEN: not set. Fine for Qwen (not gated); add it under Add-ons -> Secrets before running Llama.")

GPUs:
0, Tesla T4, 15360 MiB, 580.178.04
1, Tesla T4, 15360 MiB, 580.178.04

Internet: OK
HF_TOKEN: not set. Fine for Qwen (not gated); add it under Add-ons -> Secrets before running Llama.


## 3. Install BFCL and vLLM (about 3-5 minutes)
Everything goes into a separate Python 3.11 environment, so Kaggle's own packages are not changed.
- `bfcl-eval[oss_eval_vllm]` installs vLLM 0.8.5, the version BFCL pins for local models. vLLM 0.8.5 still supports T4 GPUs.
- `transformers`, `openai`, `fastapi` and `pydantic` are pinned to versions from vLLM 0.8.5's release period, so its server runs on the library versions it was built for.
- `setuptools` is added because Triton (used by vLLM's attention kernels) imports it, and a `uv` environment does not include it. vLLM only installs it itself on Python 3.12+.
- `soundfile` is added because BFCL imports `qwen-agent`, which needs it but does not install it.

In [3]:
!pip install -q uv
shutil.rmtree(VENV, ignore_errors=True)
!uv venv {VENV} --python 3.11
!uv pip install --python {VENV}/bin/python "bfcl-eval[oss_eval_vllm]=={BFCL_VERSION}" "transformers==4.51.3" "openai>=1.86.0,<2" "fastapi<0.116" "pydantic<2.12" "setuptools>=77.0.3,<80" soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 82.4 MB/s eta 0:00:00
Using CPython 3.11.17
Creating virtual environment at: /root/bfcl-venv
Activate with: source /root/bfcl-venv/bin/activate
Using Python 3.11.17 environment at: /root/bfcl-venv
Resolved 213 packages in 1.83s
Prepared 213 packages in 43.14s
Installed 213 packages in 746ms
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + airportsdata==20260905
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==1.11.0
 + anyio==4.15.1
 + argcomplete==3.7.2
 + astor==0.8.1
 + attrs==26.1.0
 + beautifulsoup4==4.15.0
 + bfcl-eval==2026.3.23
 + black==26.5.1
 + blake3==1.0.10
 + boto3==1.43.108
 + botocore==1.43.108
 + cachetools==7.2.0
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + cohere==5.18.0
 + compressed-tensors==0.9.3
 + cryptography==50.0.2
 + cupy-cuda12x==13.6.0
 + dashscope==1.27.7
 + datamodel-code-genera

## 4. Check the installation and the model names
Records the package versions and GPU in `run_info.json` and confirms that every name in `MODELS` exists in this BFCL version.

In [4]:
CHECK_ENV = r"""
import json, sys, importlib.metadata as md
import torch
from bfcl_eval.constants.model_config import MODEL_CONFIG_MAPPING

models, serve_model, out_path = json.loads(sys.argv[1]), sys.argv[2], sys.argv[3]
info = {"python": sys.version.split()[0]}
problems = []
for pkg in ["bfcl-eval", "vllm", "torch", "transformers", "openai"]:
    try:
        info[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        info[pkg] = "not installed"
        problems.append(f"{pkg} is not installed. Check the output of step 3.")
info["gpus"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
info["display_names"] = {}

# vLLM's attention kernels need Triton; it fails to import if setuptools is missing
try:
    import triton
except Exception as e:
    problems.append(f"triton cannot be imported ({e!r}). Check the output of step 3.")

if not info["gpus"]:
    problems.append("PyTorch cannot see a GPU.")
for m in models:
    cfg = MODEL_CONFIG_MAPPING.get(m)
    if cfg is None:
        problems.append(f"'{m}' is not a model name in this BFCL version (run `bfcl models` to list them).")
    elif cfg.model_name != serve_model:
        problems.append(f"'{m}' uses {cfg.model_name}, but the server will load {serve_model}.")
    else:
        info["display_names"][m] = cfg.display_name

print(json.dumps(info, indent=2))
with open(out_path, "w") as f:
    json.dump(info, f, indent=2)
if problems:
    print("PROBLEMS:")
    for p in problems:
        print(" -", p)
    sys.exit(1)
"""

with open(f"{WORK}/check_env.py", "w") as f:
    f.write(CHECK_ENV)

r = subprocess.run(
    [f"{VENV}/bin/python", f"{WORK}/check_env.py", json.dumps(MODELS), SERVE_MODEL, f"{WORK}/run_info.json"],
    capture_output=True, text=True,
)
print(r.stdout[-5000:])
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise RuntimeError("Environment check failed (see above).")

# Exact versions of every installed package, for the record
freeze = subprocess.run(["uv", "pip", "freeze", "--python", f"{VENV}/bin/python"], capture_output=True, text=True)
with open(f"{WORK}/requirements-freeze.txt", "w") as f:
    f.write(freeze.stdout)
print(f"Saved {len(freeze.stdout.splitlines())} package versions to {WORK}/requirements-freeze.txt")

{
  "python": "3.11.17",
  "bfcl-eval": "2026.3.23",
  "vllm": "0.8.5",
  "torch": "2.6.0",
  "transformers": "4.51.3",
  "openai": "1.109.1",
  "gpus": [
    "Tesla T4",
    "Tesla T4"
  ],
  "display_names": {
    "Qwen/Qwen3-4B-Instruct-2507": "Qwen3-4B-Instruct-2507 (Prompt)",
    "Qwen/Qwen3-4B-Instruct-2507-FC": "Qwen3-4B-Instruct-2507 (FC)"
  }
}

Saved 213 package versions to /kaggle/working/bfcl/requirements-freeze.txt


## 5. Start the vLLM server (about 3-6 minutes)
BFCL normally starts vLLM itself, but it always asks for BF16, which a T4 cannot run. So the server is started here in FP16, and BFCL is told to use it (`--skip-server-setup`). The server log is saved to `vllm_server.log`.

In [5]:
LOG_PATH = f"{WORK}/vllm_server.log"


def server_is_up():
    try:
        return requests.get(f"http://localhost:{PORT}/v1/models", timeout=5).status_code == 200
    except requests.exceptions.RequestException:
        return False


def tail(path, n=40):
    with open(path, errors="replace") as f:
        return "".join(f.readlines()[-n:])


if server_is_up():
    print("A server is already running on this port; reusing it.")
else:
    server_cmd = [
        f"{VENV}/bin/vllm", "serve", SERVE_MODEL,
        "--port", str(PORT),
        "--dtype", DTYPE,
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--tensor-parallel-size", str(TENSOR_PARALLEL_SIZE),
        "--trust-remote-code",
    ]
    print("Starting:", " ".join(server_cmd))
    server_env = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"])
    server = subprocess.Popen(server_cmd, stdout=open(LOG_PATH, "w"), stderr=subprocess.STDOUT, env=server_env)

    start = time.time()
    while not server_is_up():
        if server.poll() is not None:
            print(tail(LOG_PATH, 80))
            raise RuntimeError(f"vLLM server stopped with exit code {server.returncode}. Full log: {LOG_PATH}")
        if time.time() - start > 30 * 60:
            print(tail(LOG_PATH, 80))
            raise TimeoutError("vLLM server did not start within 30 minutes.")
        time.sleep(15)
        print(f"  waiting for the server... {int(time.time() - start)} s")
    print(f"Server ready after {int(time.time() - start)} s\n")
    print(tail(LOG_PATH, 12))

Starting: /root/bfcl-venv/bin/vllm serve Qwen/Qwen3-4B-Instruct-2507 --port 1053 --dtype half --max-model-len 16384 --gpu-memory-utilization 0.9 --tensor-parallel-size 1 --trust-remote-code
  waiting for the server... 15 s
  waiting for the server... 30 s
  waiting for the server... 45 s
  waiting for the server... 60 s
  waiting for the server... 75 s
  waiting for the server... 90 s
  waiting for the server... 105 s
  waiting for the server... 120 s
Server ready after 120 s

INFO 10-03 08:24:53 [launcher.py:36] Route: /score, Methods: POST
INFO 10-03 08:24:53 [launcher.py:36] Route: /v1/score, Methods: POST
INFO 10-03 08:24:53 [launcher.py:36] Route: /v1/audio/transcriptions, Methods: POST
INFO 10-03 08:24:53 [launcher.py:36] Route: /rerank, Methods: POST
INFO 10-03 08:24:53 [launcher.py:36] Route: /v1/rerank, Methods: POST
INFO 10-03 08:24:53 [launcher.py:36] Route: /v2/rerank, Methods: POST
INFO 10-03 08:24:53 [launcher.py:36] Route: /invocations, Methods: POST
INFO 10-03 08:24:53 

## 6. Sanity check: one question
Asks one simple question before the full run. If FP16 breaks the model on this GPU, the answer comes back empty or garbled, and the notebook stops here instead of wasting GPU time.

In [6]:
resp = requests.post(
    f"http://localhost:{PORT}/v1/chat/completions",
    json={
        "model": SERVE_MODEL,
        "messages": [{"role": "user", "content": "What is 17 + 25? Reply with only the number."}],
        "temperature": 0,
        "max_tokens": 512,
    },
    timeout=300,
)
resp.raise_for_status()
answer = resp.json()["choices"][0]["message"]["content"]
print("Model answered:", repr(answer))
if "42" not in answer:
    raise RuntimeError("Unexpected answer: the model may not work correctly in FP16 here. Check vllm_server.log before continuing.")
print("Sanity check passed.")

Model answered: '42'
Sanity check passed.


## 7. Generate answers (about 10-20 minutes per model name)
If this step is interrupted, run it again: BFCL keeps finished answers and only generates the missing ones.

In [7]:
!{VENV}/bin/bfcl generate --model {MODELS_ARG} --test-category {TEST_CATEGORY} --backend vllm --skip-server-setup

Generating results for ['Qwen/Qwen3-4B-Instruct-2507', 'Qwen/Qwen3-4B-Instruct-2507-FC']
Running full test cases for categories: ['irrelevance', 'multiple', 'parallel', 'parallel_multiple', 'simple_java', 'simple_javascript', 'simple_python'].
Max context length: 262144
server is ready!
Generating results for Qwen/Qwen3-4B-Instruct-2507: 100%|█| 1390/1390 [07:30<00
Max context length: 262144
server is ready!
Generating results for Qwen/Qwen3-4B-Instruct-2507-FC: 100%|█| 1390/1390 [06:46


## 8. Check that every question was answered
Counts answers per category and catches requests that failed inside BFCL (BFCL records those as "Error during inference" instead of stopping).

In [8]:
CHECK_RESULTS = r"""
import json, sys
from bfcl_eval.constants.eval_config import RESULT_PATH
from bfcl_eval.utils import (
    get_directory_structure_by_category,
    get_file_name_by_category,
    load_dataset_entry,
    parse_test_category_argument,
)

models, category_arg = json.loads(sys.argv[1]), sys.argv[2]
all_ok = True
for model in models:
    print(model)
    for cat in sorted(parse_test_category_argument([category_arg])):
        expected = {entry["id"] for entry in load_dataset_entry(cat)}
        path = (RESULT_PATH / model.replace("/", "_") / get_directory_structure_by_category(cat)
                / get_file_name_by_category(cat, is_result_file=True))
        rows = [json.loads(line) for line in open(path)] if path.exists() else []
        answered = {row["id"] for row in rows} & expected
        errors = [row for row in rows if str(row.get("result", "")).startswith("Error during inference")]
        ok = len(answered) == len(expected) and not errors
        all_ok &= ok
        print(f"  {cat:18s} {len(answered):4d}/{len(expected):4d} answered, "
              f"{len(errors)} failed requests   {'OK' if ok else 'CHECK'}")
        if errors:
            print("      first failure:", str(errors[0]["result"])[:300])
sys.exit(0 if all_ok else 1)
"""

with open(f"{WORK}/check_results.py", "w") as f:
    f.write(CHECK_RESULTS)

r = subprocess.run(
    [f"{VENV}/bin/python", f"{WORK}/check_results.py", json.dumps(MODELS), TEST_CATEGORY],
    capture_output=True, text=True,
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError(
        "Some questions are missing or failed.\n"
        "- Missing answers: run step 7 again (it fills in only what is missing).\n"
        "- Failed requests: read the first failure above. If it mentions the context length, "
        "raise MAX_MODEL_LEN, restart the server, and run step 7 again with --allow-overwrite."
    )
print("All questions answered.")

Qwen/Qwen3-4B-Instruct-2507
  irrelevance         240/ 240 answered, 0 failed requests   OK
  multiple            200/ 200 answered, 0 failed requests   OK
  parallel            200/ 200 answered, 0 failed requests   OK
  parallel_multiple   200/ 200 answered, 0 failed requests   OK
  simple_java         100/ 100 answered, 0 failed requests   OK
  simple_javascript    50/  50 answered, 0 failed requests   OK
  simple_python       400/ 400 answered, 0 failed requests   OK
Qwen/Qwen3-4B-Instruct-2507-FC
  irrelevance         240/ 240 answered, 0 failed requests   OK
  multiple            200/ 200 answered, 0 failed requests   OK
  parallel            200/ 200 answered, 0 failed requests   OK
  parallel_multiple   200/ 200 answered, 0 failed requests   OK
  simple_java         100/ 100 answered, 0 failed requests   OK
  simple_javascript    50/  50 answered, 0 failed requests   OK
  simple_python       400/ 400 answered, 0 failed requests   OK

All questions answered.


## 9. Score the answers

In [9]:
!{VENV}/bin/bfcl evaluate --model {MODELS_ARG} --test-category {TEST_CATEGORY}

if not os.path.exists(f"{WORK}/score/data_non_live.csv"):
    raise RuntimeError("Scoring did not produce score/data_non_live.csv (see the output above).")
print("\nScores saved to", f"{WORK}/score/")

Number of models evaluated:   0%|                        | 0/2 [00:00<?, ?it/s]🦍 Model: Qwen_Qwen3-4B-Instruct-2507
🔍 Running test: simple_python
✅ Test completed: simple_python. 🎯 Accuracy: 93.75%
🔍 Running test: irrelevance
✅ Test completed: irrelevance. 🎯 Accuracy: 84.58%
🔍 Running test: simple_java
✅ Test completed: simple_java. 🎯 Accuracy: 64.00%
🔍 Running test: parallel
✅ Test completed: parallel. 🎯 Accuracy: 88.00%
🔍 Running test: multiple
✅ Test completed: multiple. 🎯 Accuracy: 91.00%
🔍 Running test: parallel_multiple
✅ Test completed: parallel_multiple. 🎯 Accuracy: 89.00%
🔍 Running test: simple_javascript
✅ Test completed: simple_javascript. 🎯 Accuracy: 74.00%
Number of models evaluated:  50%|████████        | 1/2 [00:00<00:00,  4.36it/s]🦍 Model: Qwen_Qwen3-4B-Instruct-2507-FC
🔍 Running test: simple_python
✅ Test completed: simple_python. 🎯 Accuracy: 95.25%
🔍 Running test: irrelevance
✅ Test completed: irrelevance. 🎯 Accuracy: 88.75%
🔍 Running test: simple_java
✅ Test complete

## 10. Compare with the published leaderboard
Downloads the leaderboard's non-live scores and lines them up with this run. The leaderboard was produced in BF16 on newer GPUs, so differences of a point or two are normal. Within 3 points in every category counts as a successful reproduction.

In [10]:
import pandas as pd

COLUMNS = [
    "Non-Live Overall Acc", "Python Simple AST", "Java Simple AST", "JavaScript Simple AST",
    "Multiple AST", "Parallel AST", "Parallel Multiple AST", "Irrelevance Detection",
]
TOLERANCE = 3.0  # percentage points
LEADERBOARD_URL = "https://gorilla.cs.berkeley.edu/data_non_live.csv"


def pct(value):
    try:
        return float(str(value).strip().rstrip("%"))
    except ValueError:
        return float("nan")


ours = pd.read_csv(f"{WORK}/score/data_non_live.csv")
board = pd.read_csv(LEADERBOARD_URL)
board.to_csv(f"{WORK}/leaderboard_non_live_{date.today()}.csv", index=False)
with open(f"{WORK}/run_info.json") as f:
    info = json.load(f)

rows = []
for display_name in info["display_names"].values():
    mine = ours[ours["Model"] == display_name]
    ref = board[board["Model"] == display_name]
    if mine.empty:
        raise RuntimeError(f"{display_name} is missing from score/data_non_live.csv")
    if ref.empty:
        print(f"Note: {display_name} is not on the current leaderboard, so it has no reference score.")
    for col in COLUMNS:
        o = pct(mine.iloc[0][col])
        ref_value = pct(ref.iloc[0][col]) if not ref.empty else float("nan")
        rows.append({
            "Model": display_name,
            "Category": col,
            "Ours (FP16, T4)": o,
            "Leaderboard": ref_value,
            "Difference": round(o - ref_value, 2),
        })

comparison = pd.DataFrame(rows)
comparison.to_csv(f"{WORK}/comparison.csv", index=False)
display(comparison)

worst = comparison["Difference"].abs().max()
if pd.isna(worst):
    verdict = "No leaderboard reference was available, so the reproduction could not be checked."
elif worst <= TOLERANCE:
    verdict = f"Reproduced: every category is within {TOLERANCE:.0f} points of the leaderboard (largest difference: {worst:.2f} points)."
else:
    verdict = f"Check needed: at least one category differs from the leaderboard by more than {TOLERANCE:.0f} points (largest difference: {worst:.2f} points)."
print(verdict)

# Report for notes/bfcl_repro.md in the repository
fmt = lambda v: "n/a" if pd.isna(v) else f"{v:.2f}"
lines = [
    "# BFCL Reproduction (Update 2)",
    "",
    f"**Date:** {date.today()}",
    "**Goal:** check that the BFCL harness runs on Kaggle's T4 GPU in FP16 and reproduces the published leaderboard.",
    "",
    "## Setup",
    "",
    f"- **Models:** {', '.join(f'`{m}`' for m in MODELS)}",
    f"- **Benchmark:** BFCL `{TEST_CATEGORY}` categories, bfcl-eval {info['bfcl-eval']}",
    f"- **Inference:** vLLM {info['vllm']} on {', '.join(info['gpus'])}; FP16 (T4 has no BF16); BFCL default temperature (0.001)",
    f"- **Software:** Python {info['python']}, torch {info['torch']}, transformers {info['transformers']}",
    f"- **Reference:** {LEADERBOARD_URL} (downloaded {date.today()})",
    "",
    "## Results (accuracy, %)",
    "",
    "| Model | Category | Ours (FP16, T4) | Leaderboard | Difference |",
    "|---|---|---:|---:|---:|",
]
for row in rows:
    lines.append(f"| {row['Model']} | {row['Category']} | {fmt(row['Ours (FP16, T4)'])} | "
                 f"{fmt(row['Leaderboard'])} | {fmt(row['Difference'])} |")
lines += ["", f"**Verdict:** {verdict}", "", "## Notes", "", "- "]
report = "\n".join(lines) + "\n"
with open(f"{WORK}/bfcl_repro.md", "w") as f:
    f.write(report)
print(f"\nSaved {WORK}/bfcl_repro.md")

,Model,Category,"Ours (FP16, T4)",Leaderboard,Difference
0,Qwen3-4B-Instruct-2507 (Prompt),Non-Live Overall Acc,86.31,86.44,-0.13
1,Qwen3-4B-Instruct-2507 (Prompt),Python Simple AST,93.75,93.75,0.00
2,Qwen3-4B-Instruct-2507 (Prompt),Java Simple AST,64.00,64.00,0.00
3,Qwen3-4B-Instruct-2507 (Prompt),JavaScript Simple AST,74.00,74.00,0.00
4,Qwen3-4B-Instruct-2507 (Prompt),Multiple AST,91.00,91.00,0.00
5,Qwen3-4B-Instruct-2507 (Prompt),Parallel AST,88.00,88.00,0.00
6,Qwen3-4B-Instruct-2507 (Prompt),Parallel Multiple AST,89.00,89.50,-0.50
7,Qwen3-4B-Instruct-2507 (Prompt),Irrelevance Detection,84.58,85.00,-0.42
8,Qwen3-4B-Instruct-2507 (FC),Non-Live Overall Acc,87.81,87.88,-0.07
9,Qwen3-4B-Instruct-2507 (FC),Python Simple AST,95.25,95.50,-0.25


Reproduced: every category is within 3 points of the leaderboard (largest difference: 2.00 points).

Saved /kaggle/working/bfcl/bfcl_repro.md


## 11. Stop the server and package the results

In [11]:
srv = globals().get("server")
if srv is not None and srv.poll() is None:
    srv.terminate()
    try:
        srv.wait(timeout=60)
    except subprocess.TimeoutExpired:
        srv.kill()
    print("vLLM server stopped.")

archive = shutil.make_archive("/kaggle/working/bfcl_repro_results", "zip", WORK)
print("Saved", archive)

vLLM server stopped.
Saved /kaggle/working/bfcl_repro_results.zip


## After the run: what goes into the repository

1. Download **`bfcl_repro_results.zip`** from the notebook's **Output** section (or from the saved version's Output tab).
2. Download the notebook with its outputs (**File → Download Notebook**) and replace `notebooks/01_bfcl_reproduction.ipynb` with it.
3. From the zip:
   - `bfcl_repro.md` → `notes/bfcl_repro.md`; write a sentence or two under **Notes**
   - `comparison.csv`, `run_info.json`, `requirements-freeze.txt` and the `score/` folder → `results/bfcl_repro/`
   - The raw answers in `result/` stay out of git (`results/raw/` is ignored); keep the zip as a backup.
4. Add the Update 2 entry at the top of `UPDATES.md`, then commit and push before Wednesday's meeting.